In [1]:
import os
# 1. [关键] 必须设置缓存到数据盘 (50G硬盘保命设置)
os.environ["HF_HOME"] = "/root/autodl-tmp/hf_cache"
# 2. [关键] 关闭 HF_TRANSFER 加速 (解决 RuntimeError: no permits available)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
# 3. [关键] 关闭 Xet 加速 (解决 CAS service error)
os.environ["HF_HUB_DISABLE_XET"] = "1"
# 4. [关键] 使用国内镜像 (解决连接超时)
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import sys
sys.path.append("..")
from unsloth import FastLanguageModel
from src.utils.config_loader import load_config
import pandas as pd
import torch
from tqdm import tqdm

cfg = load_config("../configs/config.yaml")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
import subprocess

result = subprocess.run('bash -c "source /etc/network_turbo && env | grep proxy"', shell=True, capture_output=True, text=True)
output = result.stdout
for line in output.splitlines():
    if '=' in line:
        var, value = line.split('=', 1)
        os.environ[var] = value

In [3]:
# gpt-oss-20b-bnb-4bit 大约占用 14GB 显存，5090 (32GB) 完全够用
# 从已加载的配置中读取教师模型 ID
teacher_model_id = cfg.model_teacher.model_id
load_in_4bit = cfg.model_teacher.load_in_4bit


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=teacher_model_id,
    max_seq_length=2048,
    load_in_4bit=load_in_4bit,
    dtype=None
)
FastLanguageModel.for_inference(model) # [关键] 开启 Unsloth 原生推理模式，速度提升约 2 倍

==((====))==  Unsloth 2025.11.1: Fast Gpt_Oss patching. Transformers: 4.57.2.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

GptOssForCausalLM(
  (model): GptOssModel(
    (embed_tokens): Embedding(201088, 2880, padding_idx=199999)
    (layers): ModuleList(
      (0-23): 24 x GptOssDecoderLayer(
        (self_attn): GptOssAttention(
          (q_proj): Linear4bit(in_features=2880, out_features=4096, bias=True)
          (k_proj): Linear4bit(in_features=2880, out_features=512, bias=True)
          (v_proj): Linear4bit(in_features=2880, out_features=512, bias=True)
          (o_proj): Linear4bit(in_features=4096, out_features=2880, bias=True)
        )
        (mlp): GptOssMLP(
          (router): GptOssTopKRouter(
            (linear): Linear(in_features=2880, out_features=32, bias=True)
          )
          (experts): GptOssExperts(
            (gate_up_projs): ModuleList(
              (0-31): 32 x Linear4bit(in_features=2880, out_features=5760, bias=True)
            )
            (down_projs): ModuleList(
              (0-31): 32 x Linear4bit(in_features=2880, out_features=2880, bias=True)
            )


In [4]:
# Cell 3: 定义蒸馏 Prompt
# 我们给 Teacher 真实标签，让它生成“原因” (Rationale)
DISTILL_PROMPT = """
You are an expert linguist performing Aspect-Based Sentiment Analysis.

Sentence: "{text}"
Target Aspect: "{aspect}"
True Sentiment: "{polarity}"

Instructions:
1. Focus ONLY on the sentiment directed towards the target aspect: "{aspect}". Ignore other aspects in the sentence.
2. Analyze the sentence structure, specific adjectives, and context that lead to this sentiment.
3. Provide a step-by-step logical deduction.
4. Do NOT explicitly state the final sentiment label inside the explanation; only provide the reasoning.

Format your output strictly as:
<think>
[Your detailed reasoning process here]
</think>
"""

In [5]:
df_rest_train = pd.read_json("../data/processed/train_rest_clean.jsonl", lines=True)
df_lap_train = pd.read_json("../data/processed/train_lap_clean.jsonl", lines=True)

In [6]:
import re

def clean_model_output(raw_output):
    """
    清洗模型输出：
    1. 去除 <think> 之前的废话（如 analysisWe need...）
    2. 如果有多个 <think>，取最后一个（防止模型复读）
    3. 如果 </think> 缺失（被截断），自动补全
    4. 如果完全没有 <think>，尝试当做普通文本处理或返回空
    """
    if raw_output is None: return "<think>\nAnalysis failed.\n</think>"
    
    # 1. 尝试寻找 <think> 标签
    # split后取[-1]是为了应对模型输出了多个 <think> 的情况，我们通常只要最后那个正式生成的
    if "<think>" in raw_output:
        # 取最后一个 <think> 之后的内容
        content = raw_output.split("<think>")[-1]
    else:
        # 如果根本没有 <think>，说明模型完全跑偏了
        # 这种数据通常质量很差，可以选择丢弃，或者直接把全文当做 content
        # 这里为了保险，暂且保留全文，但在后续训练时可能需要筛选
        content = raw_output

    # 2. 处理结尾的 </think>
    if "</think>" in content:
        # 如果有闭合标签，取闭合标签之前的内容
        clean_content = content.split("</think>")[0]
    else:
        # [关键] 如果没有闭合标签，说明被截断了
        # 我们直接保留当前已生成的内容，因为截断的内容虽然不完整，但前面的逻辑可能还是有用的
        # 或者你也可以选择在这里 return None 丢弃这条数据
        clean_content = content

    # 3. 去除首尾空白
    clean_content = clean_content.strip()

    # 4. 重新封装成标准的 XML 格式
    return f"<think>\n{clean_content}\n</think>"

In [7]:
import json
# 设置 Padding Side 为左边
# Decoder-only 模型 (如 GPT, Llama, Qwen) 做 Batch 推理时必须左填充，
# 否则生成的全是乱码或空！
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token # 确保 pad token 存在

def generate_rationales_batch(df, dataset_name, batch_size=16):
    print(f"正在准备 {dataset_name} 的数据 (Batch Size: {batch_size})...")
    
    # --- 第一步：预处理，把所有要跑的任务展平成列表 ---
    tasks = []
    for index, row in df.iterrows():
        # 处理嵌套的 aspectTerms
        aspect_list = row.get('aspectTerms', [])
        if not isinstance(aspect_list, list): continue
            
        for aspect_item in aspect_list:
            target_term = aspect_item.get('term', '')
            target_polarity = aspect_item.get('polarity', '')
            
            # 构建 Prompt
            user_prompt = DISTILL_PROMPT.format(
                text=row['text'], 
                aspect=target_term, 
                polarity=target_polarity
            )
            
            # 存下所有必要信息
            tasks.append({
                "prompt_text": user_prompt,
                "original_row": {
                    "text": row['text'],
                    "aspect": target_term,
                    "polarity": target_polarity
                }
            })
    
    print(f"总任务数: {len(tasks)}，开始批量推理...")
    
    generated_data = []
    backup_file = f"batch_backup_{dataset_name}.jsonl"
    
    # --- 第二步：按 Batch 遍历 ---
    # range(0, 总数, 步长)
    with open(backup_file, 'w', encoding='utf-8') as f_backup:
        for i in tqdm(range(0, len(tasks), batch_size), desc="Batch Generation"):
            # 取出一个 batch 的任务
            batch_tasks = tasks[i : i + batch_size]
            batch_prompts = [t["prompt_text"] for t in batch_tasks]
            
            # 1. 构造 Chat Template (注意这里不需要 add_generation_prompt=True，因为我们要手动拼)
            # 为了 Batch 处理方便，我们直接把 prompt 当纯文本输入，或者手动 apply template
            # 这里使用 apply_chat_template 处理 list of list
            formatted_prompts = []
            for p in batch_prompts:
                msgs = [{"role": "user", "content": p}]
                # apply_chat_template 返回 string
                txt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
                formatted_prompts.append(txt)
            
            # 2. Tokenize (自动 Padding 到同一长度)
            inputs = tokenizer(
                formatted_prompts, 
                return_tensors="pt", 
                padding=True, 
                truncation=True,
                max_length=2048 # 防止极个别超长
            ).to("cuda")
            
            # 3. 批量生成
            try:
                outputs = model.generate(
                    input_ids=inputs.input_ids,
                    attention_mask=inputs.attention_mask,
                    max_new_tokens=cfg.model_teacher.max_new_tokens,
                    temperature=cfg.model_teacher.temperature,
                    use_cache=True,
                    pad_token_id=tokenizer.eos_token_id
                )
            except RuntimeError as e:
                # 显存如果爆了，这里会捕获
                print(f"\n[Error] Batch {i} failed: {e}")
                continue
            
            # 4. 批量解码 & 提取
            # 只解码新生成的部分
            # outputs 维度: [batch_size, seq_len]
            # inputs.input_ids 维度: [batch_size, prompt_len]
            # 注意：因为左填充，prompt_len 在每个样本里可能位置不一样，
            # 最简单的提取方法是 decode 全文然后 split
            
            decoded_texts = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            
            for j, full_text in enumerate(decoded_texts):
                # 对应的原始任务信息
                task_info = batch_tasks[j]
                
                # 简单粗暴提取：利用 input 的长度切片 (但在左填充下有点复杂)
                # 更稳妥的方法：直接清洗 full_text，因为 prompt 里没有 <think>，只有 output 里有
                final_rationale = clean_model_output(full_text)
                
                row_data = task_info["original_row"]
                
                # 组装样本
                new_sample = {
                    "text": row_data["text"],
                    "aspect": row_data["aspect"],
                    "polarity": row_data["polarity"],
                    # 目标输出：Teacher推理 + 最终标签
                    "target_output": f"{final_rationale}\nFinal Sentiment: {row_data['polarity']}"
                }
                
                generated_data.append(new_sample)
                f_backup.write(json.dumps(new_sample, ensure_ascii=False) + "\n")
            
            f_backup.flush()

    return generated_data

# augmented_rest_data = generate_rationales_batch(df_rest_train, "Restaurant", batch_size=16)
augmented_lap_data = generate_rationales_batch(df_lap_train, "Laptop", batch_size=16)

正在准备 Laptop 的数据 (Batch Size: 16)...
总任务数: 2358，开始批量推理...


Batch Generation: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 148/148 [1:24:06<00:00, 34.10s/it]


In [8]:
augmented_rest_data = pd.read_json("batch_backup_Restaurant.jsonl", lines=True).to_dict(orient='records') # 备份恢复
pd.DataFrame(augmented_rest_data).to_json("../data/processed/train_rest_cot_distilled.jsonl", orient='records', lines=True)
pd.DataFrame(augmented_lap_data).to_json("../data/processed/train_lap_cot_distilled.jsonl", orient='records', lines=True)
# 释放显存
del model, tokenizer
torch.cuda.empty_cache()